# Hidden Markov Model for Human Activity Recognition
## Group Project - Formative 2

This notebook implements a Hidden Markov Model to classify human activities (Standing, Walking, Jumping, Still) using accelerometer and gyroscope data from smartphone sensors.

### Background and Motivation

Human Activity Recognition (HAR) is a foundational problem in ubiquitous computing with real-world applications in health monitoring, rehabilitation, sports science, and smart device interaction. Smartphones carry high-quality inertial measurement units (IMUs) that are always available, making them ideal passive sensors for continuous activity monitoring without requiring specialist hardware.

Our unique use case targets **automated wellness monitoring**: by recognising sedentary behaviour (Still/Standing) versus active behaviour (Walking/Jumping) from a single pocket smartphone, a health application could unobtrusively log activity levels throughout the day and flag prolonged inactivity — a risk factor for cardiovascular disease and type 2 diabetes. Hidden Markov Models are particularly well-suited here because they explicitly model the *temporal transitions* between activity states (e.g., the gradual onset of walking from standing), which threshold-based or static classifiers cannot capture.

### Project Overview
- **Activities**: Standing, Walking, Jumping, Still
- **Sensors**: Accelerometer and Gyroscope (x, y, z axes)
- **Device**: iPhone 12 Pro (all group members)
- **Sampling Rate**: 20 ms (50 Hz) — consistent across all recordings
- **Total Recordings**: 120 (30 per activity)

### Group Members

| Name | Device | Sampling Rate | Recordings Contributed |
|------|--------|--------------|----------------------|
| Elissa Twizeyimana | iPhone 12    | 50 Hz | 60 |
| Uwingabire Caline | iPhone 12 Pro | 50 Hz | 60 |

### Task Allocation

| Task | Responsible Member(s) |
|------|-----------------------|
| Data collection & labelling | Elissa Twizeyimana & Uwingabire Caline |
| Feature extraction design | Elissa Twizeyimana & Uwingabire Caline |
| HMM implementation (GaussianHMM class) | Elissa Twizeyimana & Uwingabire Caline |
| Evaluation & visualisations | Elissa Twizeyimana & Uwingabire Caline |
| Report writing | Elissa Twizeyimana & Uwingabire Caline |

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import signal, stats
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Configure Data Path

Choose your environment and set the appropriate data path.

In [2]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set the path to the Data folder in your Google Drive.
# Update this to match your actual folder structure.
base_path = '/content/drive/MyDrive/Colab Notebooks/fommatives/Data'

print(f"Data path set to: {base_path}")

# Verify the data path exists and list activity folders
if os.path.exists(base_path):
    print("Data directory found.")
    activities = ['Jumping', 'Standing', 'Still', 'Walking']
    found_activities = []
    for activity in activities:
        activity_path = os.path.join(base_path, activity)
        if os.path.exists(activity_path):
            found_activities.append(activity)

    if found_activities:
        print(f"Found {len(found_activities)} activity folders: {', '.join(found_activities)}")
    else:
        print(f"Warning: No activity folders found. Expected: {', '.join(activities)}")
else:
    print(f"ERROR: Data directory not found at: {base_path}")
    print("Check your Google Drive folder structure and update base_path above.")


Mounted at /content/drive
Data path set to: /content/drive/MyDrive/Colab Notebooks/fommatives/Data
Data directory found.
Found 4 activity folders: Jumping, Standing, Still, Walking
